# 2 глава. Первичный анализ набора данных с временными рядами

Набор данных: **SKAB — Skoltech Anomaly Benchmark**  
Файл: `0.csv`  
Задача: анализ многомерного временного ряда и выявление аномалий.

## 1. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose

## 2. Загрузка и первичное знакомство с данными

In [ ]:
# Загрузка набора данных SKAB
url = "https://raw.githubusercontent.com/waico/SKAB/refs/heads/master/data/valve1/0.csv"

df = pd.read_csv(url, sep=';')

# Преобразование столбца datetime во временной формат
df['datetime'] = pd.to_datetime(df['datetime'])

# Сортировка по времени
df = df.sort_values('datetime')

# Установка datetime в качестве индекса
df.set_index('datetime', inplace=True)

# Просмотр первых строк
display(df.head())

# Информация о структуре данных
df.info()

# Основные статистические характеристики
display(df.describe())

## 3. Список числовых каналов датчиков

In [ ]:
# Список числовых каналов датчиков без служебных признаков anomaly и changepoint
sensor_cols = [
    'Accelerometer1RMS',
    'Accelerometer2RMS',
    'Current',
    'Pressure',
    'Temperature',
    'Thermocouple',
    'Voltage',
    'Volume Flow RateRMS'
]

## 4. Визуализация исходных данных

In [ ]:
# Рисунок 8. Временные ряды измерительных каналов с выделением аномалий

fig, axes = plt.subplots(len(sensor_cols), 1, figsize=(14, 12), sharex=True)

anomaly_data = df[df['anomaly'] == 1]

for ax, col in zip(axes, sensor_cols):
    ax.plot(df.index, df[col], linewidth=1, label=col)

    # Выделение аномальных наблюдений красными точками
    ax.scatter(
        anomaly_data.index,
        anomaly_data[col],
        color='red',
        s=10,
        label='Аномалия'
    )

    ax.set_ylabel(col)
    ax.grid(True)
    ax.legend(loc='upper right')

axes[0].set_title('Рисунок 8. Временные ряды измерительных каналов с выделением аномалий')
axes[-1].set_xlabel('Время')

plt.tight_layout()
plt.show()

## 5. Статистический анализ

In [ ]:
# Таблица 4. Основные статистические характеристики признаков

stats_table = df[sensor_cols].describe()

display(stats_table)

## 6. Проверка частоты дискретизации

In [ ]:
# Проверка частоты дискретизации временного ряда

df.index = pd.to_datetime(df.index)
df = df.sort_index()

time_diff = df.index.to_series().diff().dropna()

print("Минимальный интервал:", time_diff.min())
print("Максимальный интервал:", time_diff.max())
print("Наиболее частый интервал:", time_diff.mode()[0])
print("Количество разных интервалов:")
print(time_diff.value_counts())

## 7. Анализ пропущенных значений

In [ ]:
# Анализ пропущенных значений

missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    'Количество пропусков': missing_values,
    'Доля пропусков, %': missing_percent
})

display(missing_table)

## 8. Количественная оценка выбросов по правилу трёх сигм

In [ ]:
# Таблица 5. Количество выбросов по правилу трёх сигм

outliers_summary = []

for col in sensor_cols:
    mean = df[col].mean()
    std = df[col].std()

    lower_bound = mean - 3 * std
    upper_bound = mean + 3 * std

    outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outliers_share = outliers_count / len(df) * 100

    outliers_summary.append({
        'Признак': col,
        'Количество выбросов': outliers_count,
        'Доля выбросов, %': round(outliers_share, 2)
    })

outliers_df = pd.DataFrame(outliers_summary)

display(outliers_df)

## 9. Диаграммы размаха числовых признаков

In [ ]:
# Рисунок 9. Диаграммы размаха числовых признаков

fig, axes = plt.subplots(len(sensor_cols), 1, figsize=(10, 12))

for ax, col in zip(axes, sensor_cols):
    ax.boxplot(df[col], vert=False)
    ax.set_title(col)
    ax.grid(True)

fig.suptitle('Рисунок 9. Диаграммы размаха числовых признаков', y=1.01)

plt.tight_layout()
plt.show()

## 10. Анализ диапазонов значений

In [ ]:
# Рисунок 10. Диаграмма размаха диапазонов значений числовых каналов

plt.figure(figsize=(12, 6))

df[sensor_cols].boxplot()

plt.title('Рисунок 10. Диаграмма размаха диапазонов значений числовых каналов')
plt.ylabel('Значение')
plt.xticks(rotation=45)
plt.grid(True)

plt.tight_layout()
plt.show()

## 11. Корреляционный анализ

In [ ]:
# Рисунок 11. Корреляционная матрица числовых признаков

corr_matrix = df[sensor_cols].corr()

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm'
)

plt.title('Рисунок 11. Корреляционная матрица числовых признаков')
plt.tight_layout()
plt.show()

## 12. Декомпозиция временного ряда

In [ ]:
# Рисунок 12. Декомпозиция временного ряда Accelerometer1RMS

signal_col = 'Accelerometer1RMS'

# Выбор временного ряда
series = df[signal_col].astype(float)

# Декомпозиция временного ряда
# period=60 означает условную минутную сезонность при частоте около 1 секунды
decomposition = seasonal_decompose(
    series,
    model='additive',
    period=60
)

# Визуализация декомпозиции
fig = decomposition.plot()
fig.set_size_inches(10, 8)
fig.suptitle('Рисунок 12. Декомпозиция временного ряда Accelerometer1RMS', y=1.02)

plt.show()

## 13. Расчёт отношения сигнал/шум

In [ ]:
# Расчёт отношения сигнал/шум для канала Accelerometer1RMS

trend = decomposition.trend
seasonal = decomposition.seasonal
resid = decomposition.resid

# Для аддитивной модели сигнал = тренд + сезонная компонента
signal_component = trend + seasonal

# Удаление NaN, которые появляются по краям при декомпозиции
valid_data = pd.DataFrame({
    'signal': signal_component,
    'noise': resid
}).dropna()

signal_var = valid_data['signal'].var()
noise_var = valid_data['noise'].var()

snr = 10 * np.log10(signal_var / noise_var)

print(f"SNR: {snr:.2f} дБ")

## 14. Распределение остатков временного ряда

In [ ]:
# Рисунок 13. Распределение остатков временного ряда Accelerometer1RMS

plt.figure(figsize=(8, 5))

plt.hist(valid_data['noise'], bins=30, edgecolor='black')

plt.title('Рисунок 13. Распределение остатков временного ряда Accelerometer1RMS')
plt.xlabel('Остатки')
plt.ylabel('Частота')
plt.grid(True)

plt.tight_layout()
plt.show()

## 15. Итог

В ноутбуке выполнены основные этапы первичного анализа временного ряда: загрузка данных, визуализация каналов, статистический анализ, проверка частоты дискретизации, анализ пропусков и выбросов, сравнение диапазонов значений, корреляционный анализ и анализ шумовой составляющей.